## Crash Severity Label Rules

Severity is built **per crash** (one row per `CASENUM` + `YEAR_FILE`) by looking at every person and every vehicle in the crash, not just the first one.

| Class | Rule |
|---|---|
| **High** | Any person with injury code 3 (suspected serious injury) or 4 (fatal injury). |
| **Moderate** | Not high, and any person with injury code 1 (possible injury) or 2 (suspected minor injury), **or** any vehicle that was towed (`TOWED` = 2, 3, 6, 7), **or** any vehicle with disabling damage (`DEFORMED` = 6). |
| **Low** | Worst known injury is 0 (no apparent injury), no tow-away, no disabling damage, and **every** injury, tow and damage code in the crash is known. Damage must be 0, 2 or 4. |
| **Unknown** | Everything else: crashes with an unknown injury code (5, 6, 9), unknown tow code (8, 9) or unknown damage code (7, 8, 9) that could not be ruled up to moderate or high, plus 81 crashes with no person records. |

### Decisions made (team can revisit)
- A known serious or fatal injury makes a crash **high** even if another person's injury is unknown.
- `TOWED` = 3 ("towed but not due to disabling damage") counts as a tow-away.
- Functional damage (`DEFORMED` = 4) does **not** by itself make a crash moderate.
- Unknown injury, tow or damage codes are never treated as "none". If they could change the answer, the crash is labeled `unknown`.

### Notes for teammates
- `unknown` rows are **still in** the saved files. Filter them out (`SEVERITY != "unknown"`) before training or evaluating, and report how many were removed.
- CRSS's own `MAX_SEV` field was used only as a consistency check and then dropped.
- Dropped to prevent leakage: `MAX_SEV`, `NUM_INJ`, `MAXSEV_IM`, `NO_INJ_IM`, `STRATUM`, `STRATUMNAME`, `PSUSTRAT` (and their `*NAME` versions).
- **Not features:** `CASENUM`, `YEAR`, `YEAR_FILE`, `WEIGHT`, `PSU`, `PJ`, `PSU_VAR`. `WEIGHT` is for sample weighting only.
- Splits: train = 2020-2022, validation = 2023, test = 2024.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

years = [2020, 2021, 2022, 2023, 2024]
# Run with notebooks/ as the working directory.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
if NOTEBOOK_DIR.name != "notebooks" or not (PROJECT_ROOT / "data" / "README.md").is_file():
    raise RuntimeError("Run this notebook with the repository's notebooks/ directory as the working directory.")
RAW = PROJECT_ROOT / "data" / "raw"
#Claude help me build the for loop to save the headache of data upload. 
def load_table(name):
    frames = []
    for y in years:
        path = RAW / str(y) / f"{name}.csv"
        df = pd.read_csv(path, encoding="latin-1", low_memory=False)
        df.columns = df.columns.str.upper()
        df["YEAR_FILE"] = y
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

In [2]:
#Creates three tables for the three seperate CSV shapes. 
accident = load_table("accident")
vehicle = load_table("vehicle")
person = load_table("person")

In [3]:
#First step to building severeness, need to get an idea of each Injury code
person[["INJ_SEV", "INJ_SEVNAME"]].value_counts().sort_index()

INJ_SEV  INJ_SEVNAME                 
0        No Apparent Injury (O)          424761
1        Possible Injury (C)              90820
2        Suspected Minor Injury (B)       64299
3        Suspected Serious Injury (A)     34977
4        Fatal Injury (K)                  6654
5        Injured, Severity Unknown          984
6        Died Prior to Crash*                14
9        Unknown/Not Reported             23909
Name: count, dtype: int64

In [4]:
#Get rid of murky codes and get a IS_UNK code for unknown injury statuses
person["INJ_CLEAN"] = person["INJ_SEV"].replace([5, 6, 9], np.nan)
person["IS_UNK"] = person["INJ_SEV"].isin([5, 6, 9])
worst = person.groupby(["YEAR_FILE", "CASENUM"])["INJ_CLEAN"].max()
any_unk = person.groupby(["YEAR_FILE", "CASENUM"])["IS_UNK"].max()

In [5]:
#Get each Case Number to one row for the person table.
#Had to look up how to do this, Claude aided with walking me through this
inj = pd.concat([worst, any_unk], axis=1).reset_index()
inj = inj.rename(columns={"INJ_CLEAN": "WORST_INJ", "IS_UNK": "ANY_UNK"})
inj.columns

Index(['YEAR_FILE', 'CASENUM', 'WORST_INJ', 'ANY_UNK'], dtype='str')

In [6]:
#Define the injury severeness by INJ_SEV code
def injury_label(row):
    if row["WORST_INJ"] >= 3:
        return "high"
    elif row["WORST_INJ"] in [1, 2]:
        return "moderate"
    elif row["WORST_INJ"] == 0 and not row["ANY_UNK"]:
        return "low"
    else:
        return "unknown"
inj["INJ_LABEL"] = inj.apply(injury_label, axis=1)
inj["INJ_LABEL"].value_counts()

INJ_LABEL
low         115483
moderate    100664
high         34909
unknown      13524
Name: count, dtype: int64

In [7]:
#Moving onto vehicle setup and understanding the codes for each tow request. 
vehicle[["TOWED", "TOWEDNAME"]].value_counts().sort_index()

TOWED  TOWEDNAME                            
2      Towed Due to Disabling Damage             56171
3      Towed But Not Due to Disabling Damage     10017
5      Not Towed                                249406
6      Towed                                    105180
7      Towed, Unknown Reason                     10966
8      Not Reported                              31521
9      Reported as Unknown                         100
Name: count, dtype: int64

In [8]:
#Same logic as last line but now for deformitys. 
vehicle[["DEFORMED", "DEFORMEDNAME"]].value_counts().sort_index()

DEFORMED  DEFORMEDNAME                   
0         No Damage                           13939
2         Minor Damage                       110312
4         Functional Damage                   81851
6         Disabling Damage                   145090
7         Damage Reported, Extent Unknown     55205
8         Not Reported                        45054
9         Reported as Unknown                 11910
Name: count, dtype: int64

In [9]:
#Fix the murky codes and turn them into UNK.
vehicle["DMG_CLEAN"] = vehicle["DEFORMED"].replace([7, 8, 9], np.nan)
vehicle["DMG_UNK"] = vehicle["DEFORMED"].isin([7, 8, 9])
vehicle["IS_TOWED"] = vehicle["TOWED"].isin([2, 3, 6, 7])
vehicle["TOW_UNK"] = vehicle["TOWED"].isin([8, 9])
veh_flag = vehicle.groupby(["YEAR_FILE", "CASENUM"]).agg(
    WORST_DMG = ("DMG_CLEAN", "max"),
    ANY_DMG_UNK = ("DMG_UNK",   "max"),
    ANY_TOWED = ("IS_TOWED",  "max"),
    ANY_TOW_UNK = ("TOW_UNK",   "max"),
).reset_index()
veh_flag.columns

Index(['YEAR_FILE', 'CASENUM', 'WORST_DMG', 'ANY_DMG_UNK', 'ANY_TOWED',
       'ANY_TOW_UNK'],
      dtype='str')

In [10]:
#Final severity set up
#Ran into a lot of errors with this, Claude helped work through errors. Especially with the IN_LABEL = Low section. 
inj = inj.drop(columns = ["ANY_VEH_MOD", "SEVERITY"], errors="ignore")
inj = inj.merge(veh_flag, on=["YEAR_FILE", "CASENUM"], how="left")
#Creating High, Moderate, and Low crash severity based on injuries, damage, and towing. 
def final_label(row):
    if row["INJ_LABEL"] == "high":
        return "high"
    if row["INJ_LABEL"] == "moderate" or row["ANY_TOWED"] or row["WORST_DMG"] == 6: #Only disabling damage should be considered moderate. 
        return "moderate"
    # nothing serious found; low only if everything is known
    if (row["INJ_LABEL"] == "low" and not row["ANY_TOW_UNK"]
            and not row["ANY_DMG_UNK"] and row["WORST_DMG"] in [0, 2, 4]): #No damage, minor damage, and functional damage is expected in any crash even minor, as long as no tow shouldnt be greater than low. 
        return "low"
    return "unknown"
inj["SEVERITY"] = inj.apply(final_label, axis=1)
inj["SEVERITY"].value_counts()

SEVERITY
moderate    147807
low          45477
unknown      36387
high         34909
Name: count, dtype: int64

In [11]:
#Attach the severity logic to the accident table
accident = accident.merge(inj[["YEAR_FILE", "CASENUM"] + ["SEVERITY"]], on=["YEAR_FILE", "CASENUM"], how="left")
accident["SEVERITY"] = accident["SEVERITY"].fillna("unknown")
print(accident.shape)
accident["SEVERITY"].value_counts()

(264661, 82)


SEVERITY
moderate    147807
low          45477
unknown      36468
high         34909
Name: count, dtype: int64

In [12]:
#Sanity check
pd.crosstab(accident["MAX_SEV"], accident["SEVERITY"])

SEVERITY,high,low,moderate,unknown
MAX_SEV,,,,
0,0,45477,45209,32974
1,0,0,55267,0
2,0,0,45397,0
3,28827,0,0,0
4,6082,0,0,0
5,0,0,432,202
6,0,0,10,0
8,0,0,0,81
9,0,0,1492,3211


In [13]:
#Remove columns that will cause data leakage by giving away the result of the crash or any other relevant information. 
leak_cols = ["MAX_SEV", "MAX_SEVNAME", "NUM_INJ", "NUM_INJNAME", "MAXSEV_IM", "MAXSEV_IMNAME", "NO_INJ_IM", "NO_INJ_IMNAME", "STRATUM", "STRATUMNAME", "PSUSTRAT"]
accident = accident.drop(columns=leak_cols)

In [14]:
#Seperate the data into our three splits. Train 2020-2022, Validation 2023, Test 2024.
train = accident[accident["YEAR_FILE"].isin([2020, 2021, 2022])].copy()
validation = accident[accident["YEAR_FILE"] == 2023].copy()
test = accident[accident["YEAR_FILE"] == 2024].copy()
#Looking at shape, ensuring that all three have the same amount of columns. 
print(train.shape, validation.shape, test.shape)

(162900, 71) (50103, 71) (51658, 71)


In [15]:
#Looking for missing values that needs to be cleaned. Train, then Validation, then Test.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
train.isna().sum()

CASENUM                0
REGION                 0
REGIONNAME             0
PSU                    0
PJ                     0
PSU_VAR                0
URBANICITY             0
URBANICITYNAME         0
VE_TOTAL               0
VE_FORMS               0
PVH_INVL               0
PEDS                   0
PERMVIT                0
PERNOTMVIT             0
MONTH                  0
MONTHNAME              0
YEAR                   0
YEARNAME               0
DAY_WEEK               0
DAY_WEEKNAME           0
HOUR                   0
HOURNAME               0
MINUTE                 0
MINUTENAME             0
HARM_EV                0
HARM_EVNAME            0
ALCOHOL                0
ALCOHOLNAME            0
MAN_COLL               0
MAN_COLLNAME           0
RELJCT1                0
RELJCT1NAME            0
RELJCT2                0
RELJCT2NAME            0
TYP_INT                0
TYP_INTNAME            0
WRK_ZONE               0
WRK_ZONENAME      160137
REL_ROAD               0
REL_ROADNAME           0


In [16]:
validation.isna().sum()

CASENUM               0
REGION                0
REGIONNAME            0
PSU                   0
PJ                    0
PSU_VAR               0
URBANICITY            0
URBANICITYNAME        0
VE_TOTAL              0
VE_FORMS              0
PVH_INVL              0
PEDS                  0
PERMVIT               0
PERNOTMVIT            0
MONTH                 0
MONTHNAME             0
YEAR                  0
YEARNAME              0
DAY_WEEK              0
DAY_WEEKNAME          0
HOUR                  0
HOURNAME              0
MINUTE                0
MINUTENAME            0
HARM_EV               0
HARM_EVNAME           0
ALCOHOL               0
ALCOHOLNAME           0
MAN_COLL              0
MAN_COLLNAME          0
RELJCT1               0
RELJCT1NAME           0
RELJCT2               0
RELJCT2NAME           0
TYP_INT               0
TYP_INTNAME           0
WRK_ZONE              0
WRK_ZONENAME      49266
REL_ROAD              0
REL_ROADNAME          0
LGT_COND              0
LGT_CONDNAME    

In [17]:
test.isna().sum()

CASENUM               0
REGION                0
REGIONNAME            0
PSU                   0
PJ                    0
PSU_VAR               0
URBANICITY            0
URBANICITYNAME        0
VE_TOTAL              0
VE_FORMS              0
PVH_INVL              0
PEDS                  0
PERMVIT               0
PERNOTMVIT            0
MONTH                 0
MONTHNAME             0
YEAR                  0
YEARNAME              0
DAY_WEEK              0
DAY_WEEKNAME          0
HOUR                  0
HOURNAME              0
MINUTE                0
MINUTENAME            0
HARM_EV               0
HARM_EVNAME           0
ALCOHOL               0
ALCOHOLNAME           0
MAN_COLL              0
MAN_COLLNAME          0
RELJCT1               0
RELJCT1NAME           0
RELJCT2               0
RELJCT2NAME           0
TYP_INT               0
TYP_INTNAME           0
WRK_ZONE              0
WRK_ZONENAME      50731
REL_ROAD              0
REL_ROADNAME          0
LGT_COND              0
LGT_CONDNAME    

In [18]:
#WRK_ZONENAME is the only variable that is giving us trouble, looking to get better understanding of the variable. 
test[["WRK_ZONE", "WRK_ZONENAME"]].value_counts()

WRK_ZONE  WRK_ZONENAME           
4         Work Zone, Type Unknown    490
1         Construction               391
2         Maintenance                 38
3         Utility                      8
Name: count, dtype: int64

In [19]:
#When code 0 appears it gives NaN for WRK_ZONENAME, this is because 0 represent no work so we will adjust the NaN to be called "No Work Zone"
for df in [train, validation, test]:
    df.loc[df["WRK_ZONE"] == 0, "WRK_ZONENAME"] = "No Work Zone"

In [20]:
#Double check to make sure the WRK_ZONENAME fix worked. 
train.isna().sum()

CASENUM           0
REGION            0
REGIONNAME        0
PSU               0
PJ                0
PSU_VAR           0
URBANICITY        0
URBANICITYNAME    0
VE_TOTAL          0
VE_FORMS          0
PVH_INVL          0
PEDS              0
PERMVIT           0
PERNOTMVIT        0
MONTH             0
MONTHNAME         0
YEAR              0
YEARNAME          0
DAY_WEEK          0
DAY_WEEKNAME      0
HOUR              0
HOURNAME          0
MINUTE            0
MINUTENAME        0
HARM_EV           0
HARM_EVNAME       0
ALCOHOL           0
ALCOHOLNAME       0
MAN_COLL          0
MAN_COLLNAME      0
RELJCT1           0
RELJCT1NAME       0
RELJCT2           0
RELJCT2NAME       0
TYP_INT           0
TYP_INTNAME       0
WRK_ZONE          0
WRK_ZONENAME      0
REL_ROAD          0
REL_ROADNAME      0
LGT_COND          0
LGT_CONDNAME      0
SCH_BUS           0
SCH_BUSNAME       0
INT_HWY           0
INT_HWYNAME       0
WEATHER           0
WEATHERNAME       0
WKDY_IM           0
WKDY_IMNAME       0


In [21]:
validation.isna().sum()

CASENUM           0
REGION            0
REGIONNAME        0
PSU               0
PJ                0
PSU_VAR           0
URBANICITY        0
URBANICITYNAME    0
VE_TOTAL          0
VE_FORMS          0
PVH_INVL          0
PEDS              0
PERMVIT           0
PERNOTMVIT        0
MONTH             0
MONTHNAME         0
YEAR              0
YEARNAME          0
DAY_WEEK          0
DAY_WEEKNAME      0
HOUR              0
HOURNAME          0
MINUTE            0
MINUTENAME        0
HARM_EV           0
HARM_EVNAME       0
ALCOHOL           0
ALCOHOLNAME       0
MAN_COLL          0
MAN_COLLNAME      0
RELJCT1           0
RELJCT1NAME       0
RELJCT2           0
RELJCT2NAME       0
TYP_INT           0
TYP_INTNAME       0
WRK_ZONE          0
WRK_ZONENAME      0
REL_ROAD          0
REL_ROADNAME      0
LGT_COND          0
LGT_CONDNAME      0
SCH_BUS           0
SCH_BUSNAME       0
INT_HWY           0
INT_HWYNAME       0
WEATHER           0
WEATHERNAME       0
WKDY_IM           0
WKDY_IMNAME       0


In [22]:
test.isna().sum()

CASENUM           0
REGION            0
REGIONNAME        0
PSU               0
PJ                0
PSU_VAR           0
URBANICITY        0
URBANICITYNAME    0
VE_TOTAL          0
VE_FORMS          0
PVH_INVL          0
PEDS              0
PERMVIT           0
PERNOTMVIT        0
MONTH             0
MONTHNAME         0
YEAR              0
YEARNAME          0
DAY_WEEK          0
DAY_WEEKNAME      0
HOUR              0
HOURNAME          0
MINUTE            0
MINUTENAME        0
HARM_EV           0
HARM_EVNAME       0
ALCOHOL           0
ALCOHOLNAME       0
MAN_COLL          0
MAN_COLLNAME      0
RELJCT1           0
RELJCT1NAME       0
RELJCT2           0
RELJCT2NAME       0
TYP_INT           0
TYP_INTNAME       0
WRK_ZONE          0
WRK_ZONENAME      0
REL_ROAD          0
REL_ROADNAME      0
LGT_COND          0
LGT_CONDNAME      0
SCH_BUS           0
SCH_BUSNAME       0
INT_HWY           0
INT_HWYNAME       0
WEATHER           0
WEATHERNAME       0
WKDY_IM           0
WKDY_IMNAME       0


In [23]:
#Save processed data
#Claude helped me write this as I couldn't get the path to work as described by the GitHub Repo. 
OUT = PROJECT_ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
train.to_csv(OUT / "train.csv", index=False)
validation.to_csv(OUT / "validation.csv", index=False)
test.to_csv(OUT / "test.csv", index=False)

## Output and Handoff

### Files saved
Saved to `data/processed/` (local only, not committed to GitHub):

| File | Years | Rows | Purpose |
|---|---|---|---|
| `train.csv` | 2020-2022 | 162,900 | Model training and fitting preprocessing |
| `validation.csv` | 2023 | 50,103 | Hyperparameter tuning and model selection |
| `test.csv` | 2024 | 51,658 | Final evaluation only |

Each file has 71 columns, one row per crash, with the target in the `SEVERITY` column (`high`, `moderate`, `low`, `unknown`).

### Final class counts (all years, before the split)
| Class | Crashes |
|---|---|
| Moderate | 147,807 |
| Low | 45,477 |
| Unknown | 36,468 |
| High | 34,909 |

### Still to do (not done in this notebook)
- Filter `unknown` rows before training and evaluation, and report how many were removed.
- Choose features (core: roadway, environment, vehicle; expanded: plus driver-related). The `*NAME` columns are text copies of codes and can be dropped.
- Treat code columns as categories and handle unknown codes (8, 9, 98, 99).
- Fit all imputing, encoding and scaling on `train` only.
- Do not use `test.csv` for any decisions. Use `WEIGHT` only as a sample weight.